# 실습 1: Tensor와 딥러닝 데이터 핸들링 (The Foundation)

**목표:** 딥러닝의 기본 데이터 단위인 Tensor를 자유자재로 다루고, 실제 환경에 맞춰 데이터를 효율적으로 전송하는 방법을 이해한다.

## 개념 복기 및 이론 점검
1. Tensor와 NumPy 배열의 근본적인 차이점 (GPU 지원, 자동 기울기 추적 가능 여부)
2. 딥러닝에서 데이터의 형태(Shape)가 왜 중요한가 (입력 차원의 일치)
3. `dtype`과 `device` 개념

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Team-AnI/A-AND-I-4TH-AI-CODE-LAB/blob/main/2주차/lab_01_tensor_basics.ipynb)

> 위 배지를 누르면 이 노트북이 **여러분 Google 계정의 Colab**에서 열립니다. 수정본을 남기려면 `파일 → Drive에 사본 저장`.

In [ ]:
import torch
import numpy as np

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}")
print(f"MPS(Apple Silicon) 사용 가능 여부: {torch.backends.mps.is_available()}")

---
## 1. 기본 생성 및 조작

### 1-1. 다양한 방식으로 Tensor 생성하기

In [ ]:
# (1) 리스트로부터 생성
t1 = torch.tensor([[1, 2, 3], [4, 5, 6]])

# (2) NumPy 배열로부터 생성
arr = np.array([[1.0, 2.0], [3.0, 4.0]])
t2 = torch.from_numpy(arr)

# (3) 0으로 채워진 텐서
t3 = torch.zeros((2, 3, 4))

# (4) 1로 채워진 텐서
t4 = torch.ones((3, 3))

# (5) 난수 텐서 (0~1 균등분포)
t5 = torch.rand((2, 2))

# (6) 정규분포 난수
t6 = torch.randn((2, 2))

for name, t in zip(["t1", "t2", "t3", "t4", "t5", "t6"], [t1, t2, t3, t4, t5, t6]):
    print(f"{name}: shape={tuple(t.shape)}, dtype={t.dtype}")
    print(t)
    print("-" * 40)

### 1-2. Shape와 dtype 비교 분석

**확인 포인트:** 정수 리스트로 만들면 `int64`, 실수/난수는 `float32`가 기본입니다.  
딥러닝 모델의 가중치는 대부분 `float32`이므로 입력도 형변환이 필요할 수 있습니다.

In [ ]:
# dtype 변환 실습
int_tensor = torch.tensor([1, 2, 3, 4])
print(f"원본: {int_tensor}, dtype={int_tensor.dtype}")

float_tensor = int_tensor.float()        # float32로 변환
print(f"변환 후: {float_tensor}, dtype={float_tensor.dtype}")

double_tensor = int_tensor.to(torch.float64)
print(f".to() 변환: {double_tensor}, dtype={double_tensor.dtype}")

### 1-3. 인덱싱(Indexing)과 슬라이싱(Slicing) — 데이터 전처리 시뮬레이션

이미지 데이터 `(배치, 채널, 높이, 너비)` 형태를 가정한 실습입니다.

In [ ]:
# 8장의 3채널 32x32 이미지 배치를 시뮬레이션
images = torch.randn(8, 3, 32, 32)
print(f"전체 배치 shape: {images.shape}")

# (1) 첫 번째 이미지만 뽑기
first = images[0]
print(f"첫 이미지 shape: {first.shape}")

# (2) 모든 이미지의 R채널만 뽑기
red_channel = images[:, 0, :, :]
print(f"R채널만 추출 shape: {red_channel.shape}")

# (3) 이미지의 가운데 16x16 영역만 잘라내기 (Center Crop)
center_crop = images[:, :, 8:24, 8:24]
print(f"Center crop shape: {center_crop.shape}")

# (4) Shape 변형 — view / reshape / flatten
flat = images.view(8, -1)
print(f"Flatten shape (Linear 층 입력용): {flat.shape}")

---
## 2. 연산 및 장치 이동

### 2-1. 기본 연산

In [ ]:
a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.tensor([[10., 20.], [30., 40.]])

print("덧셈 (a + b):\n", a + b)
print("원소별 곱셈 (a * b):\n", a * b)
print("행렬 곱 (a @ b):\n", a @ b)
print("전치 (a.T):\n", a.T)
print("합계 (a.sum()):", a.sum().item())
print("평균 (a.mean()):", a.mean().item())

### 2-2. 브로드캐스팅(Broadcasting) 이해

shape이 달라도 일정 규칙에 따라 확장되어 연산됩니다.

In [ ]:
matrix = torch.ones((3, 4))
row = torch.tensor([1., 2., 3., 4.])
print("matrix + row =\n", matrix + row)   # row가 각 행에 더해짐

col = torch.tensor([[10.], [20.], [30.]])
print("matrix + col =\n", matrix + col)   # col이 각 열에 더해짐

### 2-3. 🔑 핵심 실습: Device 이동 (CPU ↔ GPU)

- CUDA GPU: NVIDIA GPU  
- MPS: Apple Silicon(M1/M2/M3)  
- 둘 다 없으면 CPU에서 동작

In [ ]:
# 사용 가능한 device 자동 선택
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"선택된 device: {device}")

In [ ]:
# CPU에 생성된 Tensor
cpu_tensor = torch.randn(1000, 1000)
print(f"원래 device: {cpu_tensor.device}")

# 선택된 device로 이동
moved = cpu_tensor.to(device)
print(f"이동 후 device: {moved.device}")

# 처음부터 device를 지정해서 생성할 수도 있음
direct = torch.randn(1000, 1000, device=device)
print(f"직접 생성한 텐서 device: {direct.device}")

### 2-4. ⚠️ 서로 다른 device에 있는 Tensor는 연산 불가

In [ ]:
x_cpu = torch.randn(3, 3)
x_dev = torch.randn(3, 3, device=device)

try:
    result = x_cpu + x_dev   # device 불일치
    print(result)
except RuntimeError as e:
    print("❌ RuntimeError 발생:")
    print(e)

# 해결: 한쪽을 옮겨서 device를 맞춘다
result = x_cpu.to(device) + x_dev
print(f"\n✅ device 맞춘 후 연산 성공. 결과 device: {result.device}")

### 2-5. 속도 비교 (CPU vs Device)
큰 행렬 곱을 해 보면 GPU/MPS의 위력을 체감할 수 있습니다.

In [ ]:
import time

size = 4096

# CPU
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)
start = time.time()
_ = a_cpu @ b_cpu
print(f"CPU 연산 시간: {time.time() - start:.3f}초")

# Device (GPU or MPS)
if device.type != "cpu":
    a_dev = torch.randn(size, size, device=device)
    b_dev = torch.randn(size, size, device=device)
    # 워밍업
    _ = a_dev @ b_dev
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    _ = a_dev @ b_dev
    if device.type == "cuda":
        torch.cuda.synchronize()
    print(f"{device} 연산 시간: {time.time() - start:.3f}초")

---
## ✅ 학습 결과 정리 (Verification)

- Tensor 연산 결과가 수학적 값과 일치함을 확인했다.
- `device`를 명시적으로 제어하여 CPU/GPU 환경에서 코드가 정상적으로 작동하는 것을 확인했다.
- 서로 다른 device의 Tensor는 함께 연산할 수 없다는 제약을 눈으로 확인했다.

### 🎯 핵심 결론
데이터를 다룰 때는 **'무엇'**뿐 아니라 **'어디에(device)'** 있는지가 중요하다.  
이는 자원 관리와 성능 최적화의 출발점이다.